# Twitch Sponsorship Optimization Agent
## Component 3 -- Optimization Engine

**Course:** MGMT 590-037 -- AI-Enhanced Optimization -- Daniels School of Business
**Component owner:** Matt Foor
**Pipeline position:** Data -> Prediction -> **Optimization** -> Explanation

This notebook completes every item in **Component 3** of the project checklist.
It loads the problem spec produced by Component 2, validates the schema, runs
the non-convex NLP using basin-hopping with SLSQP as the local minimizer,
validates results with dual annealing, verifies non-convexity empirically,
solves the discrete segment-selection variant with PuLP, and provides a
benchmark test that runs the optimizer from external parameters without invoking
the prediction pipeline.

**Required packages:**
```
pip install numpy scipy pulp cvxpy matplotlib pandas
```

---
## Setup -- Cell 1 (all imports and configuration)

Per the team notebook rule, every import lives in this single cell.

In [ ]:
# ============================== CELL 1: ALL IMPORTS ==============================
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from scipy.optimize import (
    basinhopping,
    dual_annealing,
    minimize,
    OptimizeResult,
)

import pulp

try:
    import cvxpy as cp
    CVXPY_AVAILABLE = True
except ImportError:
    CVXPY_AVAILABLE = False

# ── Configuration ──────────────────────────────────────────────────────────────
%matplotlib inline
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# ── Spec resolver ─────────────────────────────────────────────────────────────
# Add the correct path to the top of SPEC_CANDIDATES if the default does not match.
SPEC_CANDIDATES = [
    Path("../data/outputs"),       # notebook in Scripts/, outputs one level up
    Path("data/outputs"),          # notebook at repo root
    Path("../outputs"),
    Path("outputs"),
    Path("/mnt/user-data/outputs"),
]

def resolve_dir(probe, candidates, label="directory"):
    for d in candidates:
        if (d / probe).exists():
            return d.resolve()
    searched = "\n".join(f"  - {d.resolve()}" for d in candidates)
    raise FileNotFoundError(
        f"Could not locate {probe}.\ncwd = {Path.cwd()}\nSearched:\n{searched}\n"
        f"Add the correct folder to the top of SPEC_CANDIDATES."
    )

SPEC_DIR = resolve_dir("problem_spec.json", SPEC_CANDIDATES)
OUT_DIR  = SPEC_DIR
print("Spec / output dir:", SPEC_DIR)
print("CVXPY available  :", CVXPY_AVAILABLE)

---
## 3.1 Problem Spec Schema

The problem spec is a JSON file written by Component 2. The schema defines
every parameter the optimizer needs. Validation runs on load so that downstream
cells receive a guaranteed-valid spec. The optimizer is also fully decoupled: it
accepts an externally populated spec dict without calling any prediction code,
which is required for the benchmark test in Section 3.5.

In [ ]:
def load_spec(path):
    '''Load and validate the problem spec from a JSON file.'''
    with open(path) as f:
        spec = json.load(f)
    return spec

def validate_spec(spec, budget=None):
    '''
    Validate problem spec schema and populate budget if provided.
    Returns a validated copy with budget set.
    Raises ValueError if any constraint is violated.
    '''
    s = dict(spec)
    if budget is not None:
        s["budget"] = float(budget)

    if s.get("budget") is None:
        raise ValueError("budget must be set before calling the optimizer.")
    if s["budget"] <= 0:
        raise ValueError(f"budget must be > 0, got {s['budget']}")
    if s["mode"] not in ("continuous", "discrete"):
        raise ValueError(f"mode must be 'continuous' or 'discrete', got {s['mode']}")
    if s["objective"] not in ("conversions", "revenue"):
        raise ValueError(f"objective must be 'conversions' or 'revenue'.")
    if s["min_spend"] < 0:
        raise ValueError("min_spend must be >= 0.")

    for seg_id, params in s["curve_params"].items():
        ft = params.get("fit_type", "hill")
        if ft == "hill":
            for key in ("V", "K", "n"):
                if params.get(key, -1) <= 0:
                    raise ValueError(
                        f"Segment {seg_id}: Hill param {key} must be > 0."
                    )
        elif ft == "log":
            for key in ("a", "b"):
                if params.get(key, -1) <= 0:
                    raise ValueError(
                        f"Segment {seg_id}: log fallback param {key} must be > 0."
                    )

    for seg_id, cap in s["caps"].items():
        if cap < 0:
            raise ValueError(f"Segment {seg_id}: cap must be >= 0.")

    return s

raw_spec = load_spec(SPEC_DIR / "problem_spec.json")
spec = validate_spec(raw_spec, budget=20_000)

print("Spec loaded and validated.")
print(f"  Segments   : {spec['segments']}")
print(f"  Budget     : ${spec['budget']:,.0f}")
print(f"  Min spend  : ${spec['min_spend']:,.0f}")
print(f"  Objective  : {spec['objective']}")
print(f"  Mode       : {spec['mode']}")
print(f"  v weight   : {spec['value_weight']:.4f}")
print("\nCurve parameters:")
for seg, p in spec["curve_params"].items():
    ft = p.get("fit_type", "hill")
    if ft == "hill":
        n_flag = "  <<< FLAG: n hit upper bound (n=20, poorly identified)" \
                 if abs(p["n"] - 20.0) < 0.05 else ""
        print(f"  Seg {seg}: V={p['V']:8.3f}  K={p['K']:9.2f}  "
              f"n={p['n']:.4f}  cap=${spec['caps'][seg]:,.0f}{n_flag}")
    else:
        print(f"  Seg {seg}: LOG fallback  a={p['a']:.3f}  b={p['b']:.5f}  "
              f"cap=${spec['caps'][seg]:,.0f}")

In [ ]:
# ── External spec population (prediction pipeline bypassed) ───────────────────
# The optimizer can be driven entirely by a manually constructed spec.
# This is required for the benchmark test path in Section 3.5.
external_spec = {
    "budget":       10_000.0,
    "segments":     [0, 1],
    "curve_params": {
        "0": {"V": 200.0, "K": 5_000.0, "n": 2.5, "fit_type": "hill"},
        "1": {"V": 150.0, "K": 7_000.0, "n": 3.0, "fit_type": "hill"},
    },
    "caps":        {"0": 10_000.0, "1": 10_000.0},
    "min_spend":    500.0,
    "value_weight": 1.0,
    "objective":   "conversions",
    "mode":        "continuous",
}
external_spec = validate_spec(external_spec)
print("External spec validated (prediction pipeline bypassed).")
print("  This path is required for the benchmark test (Section 3.5).")

---
## 3.2 Continuous Allocation -- Non-Convex NLP (Basin-Hopping)

### Objective

The optimizer maximizes total incremental conversions
(or revenue when `objective = 'revenue'`) across all segments:

```
maximize  Σs  gs(xs)          [conversions mode]
maximize  v * Σs  gs(xs)      [revenue mode]
```

where `gs` is the Hill saturation function fitted in Component 2.
The objective is passed as a minimization target (negated).

### Solver design

**Primary -- basin-hopping with SLSQP:**
Basin-hopping systematically explores the objective surface by alternating
between random perturbations and local SLSQP solves. It is more principled than
naive multistart because it uses information from previous basins to guide
subsequent perturbations. SLSQP enforces the hard budget inequality and
per-segment bounds at each local solve. A custom `BudgetAcceptTest` pre-filters
perturbed candidates that violate the budget by more than a small tolerance
before passing them to the local solver, preventing the solver from wasting
iterations on infeasible basins.

**Validation -- dual annealing:**
Dual annealing is run with the same bounds and a penalty-augmented objective.
If both solvers return the same objective value the result is strong evidence
that the global optimum has been found.

### Minimum spend floor

A funded segment must clear the observed minimum sponsorship cost of $500.
Segments allocated below this floor are zeroed out in post-processing and the
remaining budget is redistributed in a final SLSQP pass.

In [ ]:
# ── Core curve functions ──────────────────────────────────────────────────────
def hill_curve(x, V, K, n):
    '''Hill saturation: g(x) = V * x^n / (K^n + x^n).'''
    xn = np.power(np.maximum(x, 0.0), n)
    Kn = np.power(K, n)
    return V * xn / (Kn + xn)

def log_curve(x, a, b):
    '''Logarithmic fallback: g(x) = a * log(b*x + 1).'''
    return a * np.log1p(np.maximum(b * x, 0.0))

def segment_lift(x_s, params):
    '''Compute incremental lift for one segment given its spend x_s.'''
    ft = params.get("fit_type", "hill")
    if ft == "hill":
        return hill_curve(x_s, params["V"], params["K"], params["n"])
    return log_curve(x_s, params["a"], params["b"])

def make_objective(params_list, value_weight, objective_mode):
    '''
    Return a negated objective function (for minimization).
    params_list: ordered list of curve_params dicts matching spec["segments"].
    '''
    def negated_obj(x):
        total = sum(segment_lift(xi, p) for xi, p in zip(x, params_list))
        if objective_mode == "revenue":
            return -value_weight * total
        return -total
    return negated_obj

def make_penalized_objective(params_list, value_weight, objective_mode,
                              budget, caps, penalty=1e8):
    '''Penalized objective for dual annealing (no explicit constraint support).'''
    def obj_pen(x):
        base = make_objective(params_list, value_weight, objective_mode)(x)
        pen  = penalty * max(0.0, np.sum(x) - budget) ** 2
        pen += sum(penalty * max(0.0, xi - cap) ** 2
                   for xi, cap in zip(x, caps))
        return base + pen
    return obj_pen

In [ ]:
# ── Custom accept test (pre-filters budget-violating perturbations) ────────────
class BudgetAcceptTest:
    '''
    Callable passed to basinhopping as accept_test.
    Rejects perturbed points that violate the budget by more than a small
    tolerance before SLSQP is called on them.
    '''
    def __init__(self, budget, caps, tol=0.05):
        self.budget = budget
        self.caps   = np.asarray(caps, dtype=float)
        self.tol    = tol           # fractional tolerance on budget overshoot

    def __call__(self, **kwargs):
        x = np.asarray(kwargs["x_new"])
        budget_ok = np.sum(x) <= self.budget * (1.0 + self.tol)
        bounds_ok = np.all(x >= -1.0) and np.all(x <= self.caps * (1.0 + self.tol))
        return bool(budget_ok and bounds_ok)

# ── Min-spend floor post-processor ────────────────────────────────────────────
def apply_min_spend_floor(x_raw, spec, obj_fn):
    '''
    Zero out segments below min_spend floor, re-solve on remaining budget.
    Returns the floor-adjusted allocation vector.
    '''
    x    = np.array(x_raw, dtype=float)
    segs = list(spec["segments"])
    caps = [spec["caps"][str(s)] for s in segs]
    B    = spec["budget"]
    floor = spec["min_spend"]

    # Zero out segments below floor
    funded = [i for i, xi in enumerate(x) if xi >= floor]
    x_adj  = np.zeros_like(x)
    if not funded:
        return x_adj

    # Re-solve on remaining budget with only funded segments
    B_rem  = min(B, sum(caps[i] for i in funded))
    params = [spec["curve_params"][str(segs[i])] for i in funded]
    caps_f = [caps[i] for i in funded]
    obj_f  = make_objective(params, spec["value_weight"], spec["objective"])
    cons   = [{"type": "ineq", "fun": lambda xx: B_rem - np.sum(xx)}]
    bnds   = [(0.0, c) for c in caps_f]
    x0     = np.array([B_rem / len(funded)] * len(funded))
    res    = minimize(obj_f, x0, method="SLSQP", bounds=bnds, constraints=cons,
                      options={"maxiter": 500, "ftol": 1e-10})
    for j, i in enumerate(funded):
        x_adj[i] = max(res.x[j], 0.0)
    return x_adj

In [ ]:
# ── Main optimizer ────────────────────────────────────────────────────────────
def run_optimizer(spec, niter=200, seed=RANDOM_STATE, verbose=True):
    '''
    Run basin-hopping (primary) and dual annealing (validation) on the spec.
    Returns a results dict with allocations and objective values from both solvers.
    '''
    segs   = list(spec["segments"])
    k      = len(segs)
    B      = spec["budget"]
    caps   = [float(spec["caps"][str(s)]) for s in segs]
    params = [spec["curve_params"][str(s)] for s in segs]
    vw     = float(spec["value_weight"])
    obj_m  = spec["objective"]

    obj_fn      = make_objective(params, vw, obj_m)
    obj_pen_fn  = make_penalized_objective(params, vw, obj_m, B, caps)

    bounds_list = [(0.0, c) for c in caps]
    bounds_2col = np.array(bounds_list)         # shape (k, 2) for dual_annealing
    constraints = [{"type": "ineq", "fun": lambda x: B - np.sum(x)}]
    stepsize    = 0.15 * B                      # 15% of budget per perturbation

    minimizer_kwargs = {
        "method": "SLSQP",
        "bounds": bounds_list,
        "constraints": constraints,
        "options": {"maxiter": 500, "ftol": 1e-10},
    }
    accept_test = BudgetAcceptTest(B, caps)

    # ── Basin-hopping ─────────────────────────────────────────────────────────
    rng = np.random.default_rng(seed)
    x0  = rng.uniform(0, [min(c, B / k) for c in caps])
    x0  = x0 / x0.sum() * min(x0.sum(), B)   # start feasible

    bh_result = basinhopping(
        obj_fn, x0,
        minimizer_kwargs=minimizer_kwargs,
        niter=niter,
        stepsize=stepsize,
        accept_test=accept_test,
        seed=seed,
    )
    x_bh_raw = np.clip(bh_result.x, 0, caps)

    # ── Dual annealing ────────────────────────────────────────────────────────
    da_result = dual_annealing(
        obj_pen_fn,
        bounds=bounds_list,
        maxiter=max(1000, niter * 10),
        seed=seed,
        minimizer_kwargs={"method": "SLSQP"},
    )
    x_da_raw = np.clip(da_result.x, 0, caps)

    # ── Min-spend floor post-processing ───────────────────────────────────────
    x_bh = apply_min_spend_floor(x_bh_raw, spec, obj_fn)
    x_da = apply_min_spend_floor(x_da_raw, spec, obj_fn)

    obj_bh = -obj_fn(x_bh)
    obj_da = -obj_fn(x_da)
    winner = "basin-hopping" if obj_bh >= obj_da else "dual annealing"

    # ── Validation ────────────────────────────────────────────────────────────
    budget_ok_bh = np.sum(x_bh) <= B + 1e-6
    bounds_ok_bh = all(0 - 1e-6 <= xi <= c + 1e-6
                       for xi, c in zip(x_bh, caps))
    budget_ok_da = np.sum(x_da) <= B + 1e-6
    bounds_ok_da = all(0 - 1e-6 <= xi <= c + 1e-6
                       for xi, c in zip(x_da, caps))

    if verbose:
        print(f"Basin-hopping  obj={obj_bh:.4f}  "
              f"budget_ok={budget_ok_bh}  bounds_ok={bounds_ok_bh}")
        print(f"Dual annealing obj={obj_da:.4f}  "
              f"budget_ok={budget_ok_da}  bounds_ok={bounds_ok_da}")
        print(f"Winner: {winner}")
        gap = abs(obj_bh - obj_da)
        if gap < 0.01 * max(abs(obj_bh), 1e-9):
            print("Solvers agree (gap < 1%) -- strong evidence of global optimum.")
        else:
            print(f"FLAG: solvers disagree by {gap:.4f}. Consider increasing niter.")

    return {
        "segs":       segs,
        "x_bh":       x_bh,
        "x_da":       x_da,
        "obj_bh":     obj_bh,
        "obj_da":     obj_da,
        "winner":     winner,
        "budget":     B,
        "valid_bh":   budget_ok_bh and bounds_ok_bh,
        "valid_da":   budget_ok_da and bounds_ok_da,
    }

In [ ]:
# ── Run on actual problem spec ────────────────────────────────────────────────
print("Running optimizer on actual problem spec (budget = $20,000)...\n")
results = run_optimizer(spec, niter=200)

# ── Allocation table ──────────────────────────────────────────────────────────
rows = []
for i, s in enumerate(results["segs"]):
    p     = spec["curve_params"][str(s)]
    lift  = segment_lift(results["x_bh"][i], p)
    rev   = lift * spec["value_weight"]
    rows.append({
        "segment":     s,
        "spend_bh":    results["x_bh"][i],
        "spend_da":    results["x_da"][i],
        "lift_bh":     lift,
        "revenue_bh":  rev,
        "funded":      results["x_bh"][i] >= spec["min_spend"],
    })
alloc_df = pd.DataFrame(rows)

print("\nOptimal allocation (basin-hopping):")
print(alloc_df[["segment", "spend_bh", "lift_bh", "revenue_bh", "funded"]]
      .to_string(index=False))
print(f"\nTotal spend : ${alloc_df['spend_bh'].sum():,.2f} "
      f"(budget ${results['budget']:,.0f})")
print(f"Total lift  : {alloc_df['lift_bh'].sum():.4f} incremental conversions")
print(f"Total rev   : ${alloc_df['revenue_bh'].sum():.2f}")

# Budget sensitivity: run at three budget levels
print("\n--- Budget sensitivity ---")
print(f"{'Budget':>10} {'Obj (BH)':>10} {'Obj (DA)':>10} {'Winner':>15}")
for B_test in [10_000, 20_000, 30_000]:
    s_test = validate_spec(raw_spec, budget=B_test)
    r      = run_optimizer(s_test, niter=50, verbose=False)
    print(f"${B_test:>9,.0f} {r['obj_bh']:>10.4f} {r['obj_da']:>10.4f} "
          f"{r['winner']:>15}")

---
## 3.3 Non-Convexity Verification

### Toy two-segment demonstration

We use hand-crafted Hill parameters with n > 1 and K values that place the
S-curve inflection point inside a realistic budget range. This cleanly
illustrates the local optimum phenomenon that motivates the non-convex solver
choice. The combined 1D objective `f(x1) = gA(x1) + gB(B - x1)` is plotted
against x1 for the total toy budget.

**Limitation note on the actual fitted params.** The Hill curves fitted in
Component 2 returned K values (half-saturation spend) of $3 to $376 for
segments 0, 1, and 3, placing the S-curve inflection point well below the
observed minimum sponsorship cost of $500. In the feasible spend range all
three segments operate in their diminishing-returns (concave) regime, so the
non-convex structure is present analytically (n > 1) but the local optimum
does not materialize at realistic budget levels for those segments. Segment 2
(K = $33,888, n = 20) is the only segment with a K value in the observable
range, but its n value hit the upper fitting bound and its parameters are
flagged as unreliable. This limitation is documented in the final report.
The toy demonstration below provides the empirical justification for the
non-convex NLP framing and the solver design.

In [ ]:
# ── Toy params: two S-shaped segments with inflections in [0, B_toy] ──────────
TOY_PARAMS = {
    "A": {"V": 200.0, "K": 5_000.0, "n": 2.5, "fit_type": "hill"},
    "B": {"V": 150.0, "K": 7_000.0, "n": 3.0, "fit_type": "hill"},
}
B_toy = 10_000.0

def g_toy(x, key):
    p = TOY_PARAMS[key]
    return hill_curve(x, p["V"], p["K"], p["n"])

# Inflection points: x_inf = K * ((n-1)/(n+1))^(1/n)
for key, p in TOY_PARAMS.items():
    x_inf = p["K"] * ((p["n"] - 1) / (p["n"] + 1)) ** (1 / p["n"])
    print(f"Seg {key}: V={p['V']:.0f}  K={p['K']:.0f}  n={p['n']}  "
          f"inflection at x = ${x_inf:,.0f}")

# ── 1D combined objective ─────────────────────────────────────────────────────
x1_grid = np.linspace(0, B_toy, 2000)
f_grid  = g_toy(x1_grid, "A") + g_toy(B_toy - x1_grid, "B")

# Local and global optima
i_bnd_lo = np.argmax(f_grid[:100])               # near x1=0 boundary
i_global  = np.argmax(f_grid)
x1_local  = x1_grid[i_bnd_lo];  f_local  = f_grid[i_bnd_lo]
x1_global = x1_grid[i_global]; f_global = f_grid[i_global]

print(f"\nLocal max (near x1=0): x1=${x1_local:,.0f}  f={f_local:.2f}")
print(f"Global max           : x1=${x1_global:,.0f}  f={f_global:.2f}")

# ── SLSQP from a low-spend start ─────────────────────────────────────────────
def obj_toy_1d(x1_arr):
    x1 = float(x1_arr[0])
    return -(g_toy(x1, "A") + g_toy(B_toy - x1, "B"))

slsqp_low = minimize(obj_toy_1d, x0=[100.0],
                     method="SLSQP",
                     bounds=[(0.0, B_toy)],
                     constraints=[{"type": "ineq", "fun": lambda x: B_toy - x[0]}],
                     options={"maxiter": 500})
x1_slsqp_low = float(slsqp_low.x[0])
f_slsqp_low  = -slsqp_low.fun
print(f"\nSLSQP from x1=$100     -> x1=${x1_slsqp_low:,.0f}  f={f_slsqp_low:.2f}  "
      f"(local: {'yes' if f_slsqp_low < f_global - 0.1 else 'global'})")

# ── Basin-hopping on the toy 1D problem ──────────────────────────────────────
bh_toy = basinhopping(
    obj_toy_1d, x0=[100.0],
    minimizer_kwargs={
        "method": "SLSQP",
        "bounds": [(0.0, B_toy)],
        "constraints": [{"type": "ineq", "fun": lambda x: B_toy - x[0]}],
    },
    niter=200, stepsize=0.15 * B_toy, seed=RANDOM_STATE,
)
x1_bh_toy = float(bh_toy.x[0])
f_bh_toy  = -bh_toy.fun
print(f"Basin-hopping from x1=$100 -> x1=${x1_bh_toy:,.0f}  f={f_bh_toy:.2f}  "
      f"(local: {'yes' if f_bh_toy < f_global - 0.1 else 'global'})")
print(f"\nBasin-hopping improvement over SLSQP: {f_bh_toy - f_slsqp_low:.2f} "
      f"incremental conversions ({(f_bh_toy/f_slsqp_low - 1):.1%})")

In [ ]:
# ── Verification plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: combined objective curve
ax = axes[0]
ax.plot(x1_grid, f_grid, color="#4B3F72", lw=2, label="f(x1) = gA(x1) + gB(B-x1)")
ax.axvline(x1_local,  ls="--", color="#E8505B", lw=1.5,
           label=f"Local max x1=${x1_local:,.0f}  f={f_local:.1f}")
ax.axvline(x1_global, ls="--", color="#119DA4", lw=1.5,
           label=f"Global max x1=${x1_global:,.0f}  f={f_global:.1f}")
ax.scatter([x1_slsqp_low], [f_slsqp_low], color="#E8505B", s=80, zorder=5,
           label=f"SLSQP (from $100) -> ${x1_slsqp_low:,.0f}")
ax.scatter([x1_bh_toy], [f_bh_toy], color="#119DA4", s=80, marker="*", zorder=5,
           label=f"Basin-hopping -> ${x1_bh_toy:,.0f}")
ax.set_xlabel("x1: budget allocated to segment A ($)")
ax.set_ylabel("Total incremental conversions f(x1)")
ax.set_title("Combined objective -- local vs global optimum (toy params)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Right: individual curves showing S-shape and inflections
x_plot = np.linspace(0, B_toy, 500)
ax2 = axes[1]
ax2.plot(x_plot, g_toy(x_plot, "A"), color="#4B3F72", lw=2, label="gA (V=200, K=5k, n=2.5)")
ax2.plot(x_plot, g_toy(x_plot, "B"), color="#119DA4", lw=2, label="gB (V=150, K=7k, n=3.0)")
for key, color in [("A", "#4B3F72"), ("B", "#119DA4")]:
    p = TOY_PARAMS[key]
    x_inf = p["K"] * ((p["n"] - 1) / (p["n"] + 1)) ** (1 / p["n"])
    ax2.axvline(x_inf, ls=":", color=color, alpha=0.7,
                label=f"Inflection {key}: ${x_inf:,.0f}")
ax2.set_xlabel("Spend ($)"); ax2.set_ylabel("Incremental conversions")
ax2.set_title("Individual Hill curves (S-shaped, n > 1)")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.suptitle("Non-convexity verification -- toy two-segment example", fontsize=11)
plt.tight_layout(); plt.show()

# Budget range over which a local optimum exists at the boundary
print("Budget range analysis:")
print("A boundary local optimum (all-in-B) exists for budgets where")
print("gA(B) < gB(B). Scanning B from $1k to $30k:")
boundaries = []
for B_scan in np.arange(1000, 30001, 500):
    ga = g_toy(B_scan, "A"); gb = g_toy(B_scan, "B")
    if gb > ga:
        boundaries.append(B_scan)
if boundaries:
    print(f"  Local optimum (all-in-B better than all-in-A) exists "
          f"for B < ${max(boundaries):,.0f}")
else:
    print("  No boundary local optimum found -- global is always all-in-A.")

In [ ]:
# ── CVXPY non-DCP check ───────────────────────────────────────────────────────
print("CVXPY non-DCP diagnostic")
print("=" * 50)
if not CVXPY_AVAILABLE:
    print("CVXPY not installed. Install with: pip install cvxpy")
    print("Skipping DCP check.")
else:
    try:
        # Attempt to express the Hill objective for the toy two-segment problem
        x_cv = cp.Variable(2, nonneg=True)
        # Hill: V * x^n / (K^n + x^n). cp.power(x, n) with n > 1 is convex
        # but the ratio (convex / (const + convex)) is not DCP.
        p_A = TOY_PARAMS["A"]
        p_B = TOY_PARAMS["B"]
        term_A = (p_A["V"] * cp.power(x_cv[0], p_A["n"])
                  / (p_A["K"] ** p_A["n"] + cp.power(x_cv[0], p_A["n"])))
        term_B = (p_B["V"] * cp.power(x_cv[1], p_B["n"])
                  / (p_B["K"] ** p_B["n"] + cp.power(x_cv[1], p_B["n"])))
        obj  = cp.Maximize(term_A + term_B)
        prob = cp.Problem(obj, [cp.sum(x_cv) <= B_toy])
        prob.solve()
        print("CVXPY accepted the problem (unexpected -- check formulation).")
    except Exception as e:
        print(f"CVXPY rejected the problem: {type(e).__name__}")
        print(f"  {str(e)[:200]}")
        print("\nThis confirms the problem is non-DCP (non-convex).")
        print("CVXPY cannot solve Hill curves with n > 1 using its DCP ruleset.")
        print("A global search method (basin-hopping) is therefore required.")

---
## 3.4 Discrete Selection -- PuLP Integer Program

The discrete formulation decides **which segments to fund** rather than
optimizing continuous spend levels. Each segment `s` is either funded (binary
variable `z_s = 1`) at a pre-determined representative spend, or not funded
(`z_s = 0`). The objective is the Hill lift at the representative spend level
weighted by the binary variable.

Representative spend is set to 80% of each segment's cap, which represents a
high but not maximum commitment. The budget constraint ensures the sum of funded
representative spends does not exceed `B`.

This formulation is a binary knapsack solved with PuLP's CBC solver.

In [ ]:
def run_discrete_optimizer(spec):
    '''Solve the binary segment-selection problem with PuLP.'''
    segs   = spec["segments"]
    caps   = {s: float(spec["caps"][str(s)]) for s in segs}
    params = {s: spec["curve_params"][str(s)] for s in segs}
    B      = spec["budget"]
    vw     = float(spec["value_weight"])
    obj_m  = spec["objective"]

    # Representative spend: 80% of cap or max feasible given budget
    rep_spend = {s: min(0.80 * caps[s], B) for s in segs}

    # Lift at representative spend
    rep_lift = {s: segment_lift(rep_spend[s], params[s]) for s in segs}
    rep_val  = {s: vw * rep_lift[s] if obj_m == "revenue" else rep_lift[s]
                for s in segs}

    prob = pulp.LpProblem("segment_selection", pulp.LpMaximize)
    z    = {s: pulp.LpVariable(f"z_{s}", cat="Binary") for s in segs}

    # Objective
    prob += pulp.lpSum(rep_val[s] * z[s] for s in segs)

    # Budget constraint
    prob += (pulp.lpSum(rep_spend[s] * z[s] for s in segs) <= B,
             "budget_constraint")

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    status = pulp.LpStatus[prob.status]
    selected = [s for s in segs if pulp.value(z[s]) > 0.5]
    total_spend = sum(rep_spend[s] for s in selected)
    total_lift  = sum(rep_lift[s]  for s in selected)
    obj_val     = pulp.value(prob.objective)

    return {
        "status":      status,
        "selected":    selected,
        "total_spend": total_spend,
        "total_lift":  total_lift,
        "obj_value":   obj_val,
        "rep_spend":   rep_spend,
        "rep_lift":    rep_lift,
    }

disc = run_discrete_optimizer(spec)
print(f"PuLP status          : {disc['status']}")
print(f"Funded segments      : {disc['selected']}")
print(f"Total spend          : ${disc['total_spend']:,.2f}")
print(f"Total lift           : {disc['total_lift']:.4f} incremental conversions")
print(f"Objective value      : {disc['obj_value']:.4f}")

print("\nPer-segment discrete selection:")
print(f"{'Seg':>4} {'Rep spend':>11} {'Rep lift':>10} {'Selected':>10}")
for s in spec["segments"]:
    print(f"{s:>4} ${disc['rep_spend'][s]:>10,.0f} "
          f"{disc['rep_lift'][s]:>10.4f} "
          f"{'YES' if s in disc['selected'] else 'no':>10}")

---
## 3.5 Benchmark Test Readiness

The benchmark test confirms two things required by the checklist:

1. **Correctness:** the optimizer recovers the global optimum on a toy problem
   where the solution is known or tightly bounded numerically.
2. **Decoupling:** the optimizer runs correctly when the spec is populated
   externally, with no call to the prediction pipeline.

### Toy benchmark design

We use the same two-segment external spec from Section 3.1. With n > 1 in both
segments and a shared budget of $10,000, the global optimum is the corner
solution that puts all budget into the higher-value segment (Segment A, which
has higher V and its inflection point at $3,345 vs Segment B at $5,558). We
verify that basin-hopping reaches this solution and that a single SLSQP run
from a low-spend start on A does not.

In [ ]:
print("=" * 60)
print("BENCHMARK TEST -- external spec, prediction pipeline bypassed")
print("=" * 60)
print(f"\nSpec: {external_spec['segments']}  Budget=${external_spec['budget']:,.0f}")
for s in external_spec["segments"]:
    p = external_spec["curve_params"][str(s)]
    print(f"  Seg {s}: V={p['V']}  K={p['K']}  n={p['n']}")

# ── Known reference values (numerical scan) ───────────────────────────────────
B_bm    = external_spec["budget"]
segs_bm = external_spec["segments"]
params_bm = [external_spec["curve_params"][str(s)] for s in segs_bm]
obj_bm_fn = make_objective(params_bm, 1.0, "conversions")

# Scan 1D to find reference global optimum
x1_scan = np.linspace(0, B_bm, 10_000)
f_scan  = np.array([-obj_bm_fn(np.array([x1, B_bm - x1])) for x1 in x1_scan])
i_ref   = np.argmax(f_scan)
x_ref   = np.array([x1_scan[i_ref], B_bm - x1_scan[i_ref]])
f_ref   = f_scan[i_ref]
print(f"\nReference global optimum (1D scan):")
print(f"  x0=${x_ref[0]:,.0f}  x1=${x_ref[1]:,.0f}  f={f_ref:.4f}")

# ── Single SLSQP from low-spend start ────────────────────────────────────────
x0_low  = np.array([100.0, B_bm - 100.0])
slsqp_r = minimize(
    obj_bm_fn, x0_low, method="SLSQP",
    bounds=[(0, B_bm)] * 2,
    constraints=[{"type": "ineq", "fun": lambda x: B_bm - np.sum(x)}],
    options={"maxiter": 500},
)
f_slsqp_bm = -slsqp_r.fun
print(f"\nSingle SLSQP (from low-spend start):")
print(f"  x0=${slsqp_r.x[0]:,.0f}  x1=${slsqp_r.x[1]:,.0f}  f={f_slsqp_bm:.4f}")
print(f"  Trapped at local optimum: {f_slsqp_bm < f_ref - 0.01}")

# ── Basin-hopping benchmark ───────────────────────────────────────────────────
bm_result = run_optimizer(external_spec, niter=200, seed=RANDOM_STATE, verbose=False)
f_bh_bm   = bm_result["obj_bh"]
f_da_bm   = bm_result["obj_da"]
print(f"\nBasin-hopping:")
print(f"  x0=${bm_result['x_bh'][0]:,.0f}  x1=${bm_result['x_bh'][1]:,.0f}"
      f"  f={f_bh_bm:.4f}")
print(f"\nDual annealing:")
print(f"  x0=${bm_result['x_da'][0]:,.0f}  x1=${bm_result['x_da'][1]:,.0f}"
      f"  f={f_da_bm:.4f}")

# ── Pass / fail criteria ──────────────────────────────────────────────────────
tol = 0.01 * f_ref
bh_pass    = abs(f_bh_bm - f_ref) <= tol
slsqp_fail = f_slsqp_bm < f_ref - tol
print(f"\nBENCHMARK RESULTS")
print(f"  Basin-hopping recovers global optimum (within 1%): {'PASS' if bh_pass else 'FAIL'}")
print(f"  Single SLSQP trapped at local optimum            : {'PASS' if slsqp_fail else 'not trapped (check params)'}")
print(f"  Optimizer runs on external spec (no prediction)  : PASS")

---
## Save Optimization Results

In [ ]:
opt_output = {
    "budget":           spec["budget"],
    "solver_winner":    results["winner"],
    "objective_value":  results["obj_bh"],
    "total_spend":      float(np.sum(results["x_bh"])),
    "allocation": {
        str(s): {
            "spend":   float(results["x_bh"][i]),
            "lift":    float(segment_lift(results["x_bh"][i],
                              spec["curve_params"][str(s)])),
            "funded":  bool(results["x_bh"][i] >= spec["min_spend"]),
        }
        for i, s in enumerate(results["segs"])
    },
    "discrete_selection": {
        "funded_segments": disc["selected"],
        "total_spend":     disc["total_spend"],
        "total_lift":      disc["total_lift"],
    },
    "benchmark": {
        "bh_passes":   bool(bh_pass),
        "slsqp_fails": bool(slsqp_fail),
        "f_ref":       float(f_ref),
        "f_bh":        float(f_bh_bm),
        "f_slsqp":     float(f_slsqp_bm),
    },
    "solver_params": {
        "niter": 200,
        "stepsize_pct_budget": 0.15,
        "min_spend_floor": spec["min_spend"],
        "ipw_clip_pct": 99,
    },
}

out_path = OUT_DIR / "optimization_results.json"
with open(out_path, "w") as f:
    json.dump(opt_output, f, indent=2)
print("optimization_results.json written to:", out_path)
print(json.dumps(opt_output, indent=2))

---
## Component 3 Completion Summary

| Checklist item | Where satisfied | Result |
|---|---|---|
| **3.1** Problem spec schema (all required fields) | 3.1 | Defined |
| 3.1 Validate on load | 3.1 | `validate_spec()` raises on violation |
| 3.1 External spec population (bypass prediction) | 3.1 | `external_spec` demo |
| **3.2** Negated objective (minimization target) | 3.2 | `make_objective()` |
| 3.2 Budget inequality constraint | 3.2 | SLSQP constraints dict |
| 3.2 Per-segment bounds | 3.2 | SLSQP bounds list |
| 3.2 Basin-hopping with SLSQP local minimizer | 3.2 | `basinhopping(..., minimizer_kwargs)` |
| 3.2 niter >= 200 | 3.2 | niter=200 |
| 3.2 stepsize = 15% of B | 3.2 | stepsize=0.15*B |
| 3.2 Custom accept test | 3.2 | `BudgetAcceptTest` |
| 3.2 Dual annealing validation | 3.2 | `dual_annealing` with penalty |
| 3.2 Compare solvers, flag disagreement | 3.2 | Gap check printed |
| 3.2 Return x*, objective, winning solver | 3.2 | `run_optimizer()` returns dict |
| 3.2 Validate final solution | 3.2 | budget_ok and bounds_ok checks |
| 3.2 Min-spend floor post-processing | 3.2 | `apply_min_spend_floor()` |
| **3.3** 1D combined objective plot | 3.3 | Toy two-segment plot |
| 3.3 Local vs global optimum shown | 3.3 | Marked on plot |
| 3.3 SLSQP trapped vs basin-hopping escape | 3.3 | Scatter markers on plot |
| 3.3 Budget range for local optima | 3.3 | Scan printed |
| 3.3 CVXPY non-DCP check | 3.3 | Exception caught and reported |
| **3.4** Binary decision variables per segment | 3.4 | PuLP LpVariable Binary |
| 3.4 Maximize lift * z | 3.4 | PuLP objective |
| 3.4 Budget constraint | 3.4 | PuLP constraint |
| 3.4 PuLP CBC solver | 3.4 | `PULP_CBC_CMD` |
| 3.4 Return selected set and objective | 3.4 | `disc` dict |
| **3.5** Benchmark on toy problem | 3.5 | Two-segment external spec |
| 3.5 Basin-hopping recovers global optimum | 3.5 | PASS check |
| 3.5 SLSQP trapped at local optimum | 3.5 | PASS check |
| 3.5 Optimizer runs on external spec | 3.5 | PASS -- no prediction call |

**Handoff to Component 4 (Explanation and Validation).**
`optimization_results.json` contains the winning allocation, objective value,
funded segments, discrete selection result, and benchmark pass/fail flags.
The plain-language explanation layer in Component 4 should reference the
allocation, the funded segment curve shapes (S-shaped vs concave, inflection
relative to allocated spend), and the budget sensitivity results.